# Chapter 6 — Top-down Ontology Development
### Notebook 1 · Foundational ontologies

*Book reference: Section 6.1*

A foundational ontology is a set of very general categories agreed in advance, so that domain modellers make the same distinctions the same way. Its real product is not the categories — it is the **questions** that place things in them.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch06_toolkit as ch6
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

## 1. The categories

A DOLCE-flavoured tree, simplified to seven leaves. The `branch` column carries the oldest distinction in the subject: **endurants** exist *through* time (a giraffe is wholly present at every moment it exists), **perdurants** unfold *in* time (a hunt has temporal parts).

In [ ]:
print(pd.DataFrame([{'id': c.id, 'name': c.name, 'branch': c.branch,
                     'examples': ', '.join(c.examples)}
                    for c in ch6.CATEGORIES]).to_string(index=False))

In [ ]:
for c in ch6.CATEGORIES:
    print(f'{c.name} ({c.branch})')
    print(f'   {c.gloss}')
    print(f'   e.g. {", ".join(c.examples)}\n')

## 2. The decision questions

This is the part that makes alignment an engineering activity. Each question is a yes/no test with a defensible answer for every category.

In [ ]:
for key, question in ch6.DECISION_QUESTIONS.items():
    print(f'  {key:11s} {question}')

In [ ]:
rows = []
for c in ch6.CATEGORIES:
    row = {'category': c.name}
    row.update({k: ('yes' if v else 'no')
                for k, v in ch6.category_answers(c.id).items()})
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

## 3. Aligning a class by interrogation

`identify_category` takes whatever answers you have so far and returns the categories still consistent with them. Watch the candidate set shrink.

In [ ]:
steps = [{}, {'happens': True}, {'happens': True, 'telic': True}]
for answers in steps:
    print(f'{str(answers):46s} -> {ch6.identify_category(answers)}')

In [ ]:
print('Aligning "clay":')
answers = {}
for question, answer in [('happens', False), ('spatial', True), ('mass', True)]:
    answers[question] = answer
    print(f'  {question}={answer}  -> candidates {ch6.identify_category(answers)}')
assert ch6.identify_category(answers) == ['amount-of-matter']

In [ ]:
print('Aligning "the colour of a leaf":')
answers = {}
for question, answer in [('happens', False), ('spatial', False), ('dependent', True)]:
    answers[question] = answer
    print(f'  {question}={answer}  -> candidates {ch6.identify_category(answers)}')
assert ch6.identify_category(answers) == ['quality']

> **Three questions, one answer.** That is what a foundational ontology buys: not a list of categories to memorise but a short, repeatable interrogation that two modellers will answer the same way. Notebook 4 hands the interrogation to an agent — and derives the optimal order of questions from a reward function.

## 4. DOLCE is not the only choice

BFO draws many of the same distinctions differently, and the mismatch is a real project decision rather than a detail.

In [ ]:
print(pd.DataFrame(ch6.BFO_COMPARISON).to_string(index=False))

> The last row is the sharpest: BFO is **realist** and has no place for abstract entities at all. If your domain needs to talk about numbers, propositions or musical works as first-class things, that is not a preference — it decides the choice for you.

### Exercise 1.1 — Align four domain classes

Place `a hole in a leaf`, `digestion`, `the number seven` and `a herd` in the category tree by answering the decision questions. One of them exposes a limit of this seven-category tree — say which.

> **Hint.** Answer only the questions you need; `identify_category` accepts partial answers.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 1.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
cases = {
    'a hole in a leaf': {'happens': False, 'spatial': True, 'mass': False,
                         'dependent': True},
    'digestion':        {'happens': True, 'telic': False},
    'the number seven': {'happens': False, 'spatial': False, 'dependent': False},
}
for name, answers in cases.items():
    print(f'{name:20s} -> {ch6.identify_category(answers)}')
assert ch6.identify_category(cases['a hole in a leaf']) == ['feature']
assert ch6.identify_category(cases['digestion']) == ['process']
assert ch6.identify_category(cases['the number seven']) == ['abstract']

herd = {'happens': False, 'spatial': True, 'mass': False, 'dependent': False}
print(f"{'a herd':20s} -> {ch6.identify_category(herd)}")
print('\nA herd comes out as a physical object, which is not wrong but is not\n'
      'informative either: this tree has no COLLECTION category, even though the\n'
      'part-whole taxonomy in Notebook 2 needs one for member-of. Real\n'
      'foundational ontologies do have it. The lesson is that your category set\n'
      'and your relation set have to be designed together.')

### Exercise 1.2 — Is any question redundant?

Find the smallest set of questions that still distinguishes all seven categories. Then, for each question, name the pair of categories that *only* it separates.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 1.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
import itertools
questions = list(ch6.DECISION_QUESTIONS)
best = None
for size in range(1, len(questions) + 1):
    for subset in itertools.combinations(questions, size):
        vectors = {c.id: tuple(ch6.category_answers(c.id)[q] for q in subset)
                   for c in ch6.CATEGORIES}
        if len(set(vectors.values())) == len(ch6.CATEGORIES):
            best = subset
            break
    if best:
        break
print('smallest sufficient question set:', best)
print('droppable:', [q for q in questions if q not in best] or 'none')
assert best is not None and len(best) == len(questions)

print('\nfor each question, the pair only it separates:')
for q in questions:
    others = [x for x in questions if x != q]
    collisions = {}
    for c in ch6.CATEGORIES:
        key = tuple(ch6.category_answers(c.id)[x] for x in others)
        collisions.setdefault(key, []).append(c.id)
    merged = [v for v in collisions.values() if len(v) > 1]
    print(f'  {q:11s} -> {merged}')
print('\nNo question is redundant: drop any one and two categories collapse into\n'
      'each other. That is a well-designed question set -- and it does NOT mean\n'
      'every alignment needs all five. Notebook 4 shows the optimal policy\n'
      'settling most classes in three, because a question is only asked on the\n'
      'branch where it still discriminates.')